# Binary **joy** vs rest — Colab repro

Branch: **`emotion-rec-joy`**. Mount Drive, GPU runtime, set `HF_TOKEN` for Qwen.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, subprocess
REPO = "/content/drive/MyDrive/DS-GA-3001-Data-Engineering-Project"
ROOT = os.path.join(REPO, "emotions_rec")
os.chdir(ROOT)
sys.path.insert(0, os.path.join(ROOT, "src"))
print("cwd:", os.getcwd())

In [ ]:
!python scripts/smoke_check_emotions_rec.py

In [ ]:
!python scripts/prepare_emotions_binary.py --label joy

In [ ]:
import subprocess
TRAIN = "data/processed/emotions_joy_smoke_train"
VAL = "data/processed/emotions_joy_smoke_validation.csv"
cmd = [
    "python", "-u", "src/main_cluster_emotion_binary.py",
    "-sample_size", "200", "-filename", TRAIN, "-val_path", VAL,
    "-balance", "False", "-sampling", "thompson", "-filter_label", "True",
    "-model_finetune", "bert-base-uncased", "-labeling", "qwen", "-model", "text",
    "-baseline", "0.10", "-metric", "f1_pos", "-cluster_size", "8",
    "-positive_label", "joy",
    "-hf_model_id", "Qwen/Qwen2.5-3B-Instruct",
    "-max_iterations", "3", "-num_train_epochs", "2",
    "-max_length", "128", "-batch_size", "16", "-confidence_threshold", "0.40",
]
subprocess.run(cmd, cwd=ROOT, check=True)

In [ ]:
# Set CHECKPOINT to a folder under models/ that contains config.json (best eval_f1_pos from logs).
CHECKPOINT = os.path.join(ROOT, "models", "binary_joy_fine_tunned_0_bandit_0")
subprocess.run([
    "python", "-u", "src/eval_emotion_binary.py",
    "-test_path", "data/processed/emotions_joy_test.csv",
    "-model_path", CHECKPOINT,
    "-target_emotion", "joy",
    "-base_model", "bert-base-uncased", "-max_length", "128",
], cwd=ROOT, check=True)